In [1]:
import time
import torch
import numpy as np
from torch.utils.data import DataLoader
from sentence_transformers import InputExample, SentenceTransformer
from sentence_transformers.cross_encoder import CrossEncoder
from sklearn.metrics.pairwise import cosine_similarity

# Verify GPU availability
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")
if device == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")

Using device: cuda
GPU: NVIDIA GeForce RTX 5090


In [2]:
persona = "Juliette is a 38-year-old female working as a service or sales worker. They have secondary or less level education and good health. They live as: living alone. They own a car."

# Ground-truth scores assigned by Gemini
training_data = [
    # routine-005: Full-time work, realistic commute/errands, evening at home.
    InputExample(texts=[persona, "HOME 00:00-07:30, WORK 07:30-18:00, OTHER 18:00-19:30, HOME 19:30-24:00"], label=0.90),
    
    # routine-007: Part-time/half-day work, extended time at home.
    InputExample(texts=[persona, "HOME 00:00-08:00, WORK 08:00-12:00, OTHER 12:00-14:00, HOME 14:00-24:00"], label=0.40),
    
    # routine-030: Highly fragmented, only 2 hours of work. Fits an erratic weekend, not a typical weekday.
    InputExample(texts=[persona, "HOME 00:00-07:00, WORK 07:00-09:00, OTHER 09:00-10:00, OTHER 10:00-12:00, HOME 12:00-13:00, OTHER 13:00-15:00, HOME 15:00-16:00, OTHER 16:00-18:00, HOME 18:00-24:00"], label=0.20),
    
    # routine-025: Massive mid-day errand block, almost no work.
    InputExample(texts=[persona, "HOME 00:00-08:30, WORK 08:30-10:00, OTHER 10:00-12:30, OTHER 12:30-15:00, HOME 15:00-24:00"], label=0.10),
    
    # routine-002: 24 hours at home. Impossible for a healthy on-site worker on a weekday.
    InputExample(texts=[persona, "HOME 00:00-24:00"], label=0.00)
]

# Create a DataLoader for the training loop
train_dataloader = DataLoader(training_data, shuffle=True, batch_size=2)
print("Dataset loaded successfully.")

Dataset loaded successfully.


In [3]:
model_path = '/home/gustavo/citybehavex/notebooks/04_reranker_testing/modernbert-schedule-aligner'

In [5]:
print("Loading ModernBERT Cross-Encoder...")

# Initialize the model for regression (num_labels=1)
# Using flash_attention_2 to maximize the RTX 5090 throughput
cross_encoder = CrossEncoder(
    'answerdotai/ModernBERT-base', 
    num_labels=1, 
    max_length=256, 
)

print("Starting fine-tuning...")
# Train the model. Because num_labels=1, MSELoss is used automatically.
cross_encoder.fit(
    train_dataloader=train_dataloader,
    epochs=10, # Kept high for this tiny toy dataset to force convergence
    warmup_steps=2,
    output_path=model_path
)

cross_encoder.save(model_path)

print(f"Fine-tuning complete. Model saved to {model_path} .")

Loading ModernBERT Cross-Encoder...


Loading weights:   0%|          | 0/136 [00:00<?, ?it/s]

[transformers] ModernBertForSequenceClassification LOAD REPORT from: answerdotai/ModernBERT-base
Key               | Status     | 
------------------+------------+-
decoder.bias      | UNEXPECTED | 
classifier.bias   | MISSING    | 
classifier.weight | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': None, 'bos_token_id': None}.


Starting fine-tuning...


Step,Training Loss


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Fine-tuning complete. Model saved to /home/gustavo/citybehavex/notebooks/04_reranker_testing/modernbert-schedule-aligner .


In [6]:
# Load the fine-tuned model
import os
from sentence_transformers import CrossEncoder

# Load the fine-tuned model safely
eval_model = CrossEncoder(model_path)


# 1. Accuracy Test on the Ground Truth
test_pairs = [[ex.texts[0], ex.texts[1]] for ex in training_data]
predicted_scores = eval_model.predict(test_pairs)

print("--- Accuracy Test ---")
for i, ex in enumerate(training_data):
    print(f"Target: {ex.label:.2f} | Predicted: {predicted_scores[i]:.2f} | Schedule: {ex.texts[1][:40]}...")

# 2. Latency and Throughput Test
print("\n--- Latency & Throughput Evaluation ---")
# Generate a large synthetic batch to saturate the RTX 5090 (32GB VRAM)
# We duplicate our pairs to create a massive workload
num_samples = 10_000 
large_batch = test_pairs * (num_samples // len(test_pairs))

start_time = time.perf_counter()
# Adjust batch_size upwards if your VRAM allows; 256 is a safe starting point
_ = eval_model.predict(large_batch, batch_size=256, show_progress_bar=False)
end_time = time.perf_counter()

total_time = end_time - start_time
throughput = len(large_batch) / total_time

print(f"Processed {len(large_batch):,} pairs in {total_time:.2f} seconds.")
print(f"Throughput: {throughput:,.0f} pairs / second.")

Loading weights:   0%|          | 0/138 [00:00<?, ?it/s]

--- Accuracy Test ---
Target: 0.90 | Predicted: 0.37 | Schedule: HOME 00:00-07:30, WORK 07:30-18:00, OTHE...
Target: 0.40 | Predicted: 0.35 | Schedule: HOME 00:00-08:00, WORK 08:00-12:00, OTHE...
Target: 0.20 | Predicted: 0.30 | Schedule: HOME 00:00-07:00, WORK 07:00-09:00, OTHE...
Target: 0.10 | Predicted: 0.31 | Schedule: HOME 00:00-08:30, WORK 08:30-10:00, OTHE...
Target: 0.00 | Predicted: 0.10 | Schedule: HOME 00:00-24:00...

--- Latency & Throughput Evaluation ---
Processed 10,000 pairs in 5.23 seconds.
Throughput: 1,911 pairs / second.


In [7]:
print("Loading Bi-Encoder Embedding Model...")
# Load an embedding model
bi_encoder = SentenceTransformer('nomic-ai/modernbert-embed-base')

persona_embedding = bi_encoder.encode([persona])
schedule_texts = [ex.texts[1] for ex in training_data]
schedule_embeddings = bi_encoder.encode(schedule_texts)

# Calculate Cosine Similarity
similarities = cosine_similarity(persona_embedding, schedule_embeddings)[0]

print("\n--- Bi-Encoder Similarity Scores ---")
print("Notice how standard embeddings fail to punish logically impossible schedules (like 24h at HOME).")
for i, ex in enumerate(training_data):
    print(f"Ground Truth: {ex.label:.2f} | Bi-Encoder Score: {similarities[i]:.2f} | Schedule: {ex.texts[1][:40]}...")

Loading Bi-Encoder Embedding Model...


Loading weights:   0%|          | 0/134 [00:00<?, ?it/s]


--- Bi-Encoder Similarity Scores ---
Notice how standard embeddings fail to punish logically impossible schedules (like 24h at HOME).
Ground Truth: 0.90 | Bi-Encoder Score: 0.50 | Schedule: HOME 00:00-07:30, WORK 07:30-18:00, OTHE...
Ground Truth: 0.40 | Bi-Encoder Score: 0.50 | Schedule: HOME 00:00-08:00, WORK 08:00-12:00, OTHE...
Ground Truth: 0.20 | Bi-Encoder Score: 0.51 | Schedule: HOME 00:00-07:00, WORK 07:00-09:00, OTHE...
Ground Truth: 0.10 | Bi-Encoder Score: 0.50 | Schedule: HOME 00:00-08:30, WORK 08:30-10:00, OTHE...
Ground Truth: 0.00 | Bi-Encoder Score: 0.46 | Schedule: HOME 00:00-24:00...
